We need to do inference on the test data. Once we have the predictions, we also need to do a posthoc analysis to correctly get the tassel densities for each test image. Another thing to consider here is that we may need to report the mae's seperately for each block, and also entire dataset together.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

2025-06-26 12:21:55.261622: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-26 12:21:55.297317: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-26 12:21:55.297344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-26 12:21:55.298188: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-26 12:21:55.305088: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# load the trained model
basemodel2_stage2 = tf.keras.models.load_model("models/stage2_basemodel2.keras")

2025-06-26 12:21:57.024890: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:06:00.0, compute capability: 7.0


In [3]:
# where is the data?

In [4]:
# The input features are at some other location
test_features_location = "seq_2_seq_test_data"

In [5]:
# os.listdir(test_features_location)

In [6]:
# only get the features files
all_test_featurefile_names = [file for file in os.listdir(test_features_location) if file.split(".")[0][-14:] == 'input_features']
all_test_featurefile_names.sort()

In [7]:
all_test_featurefile_names

['block_0103_extracted_input_features.npy',
 'block_0104_extracted_input_features.npy',
 'block_0105_extracted_input_features.npy',
 'block_0106_extracted_input_features.npy',
 'block_0201_extracted_input_features.npy',
 'block_0202_extracted_input_features.npy',
 'block_0205_extracted_input_features.npy',
 'block_0206_extracted_input_features.npy',
 'block_0302_extracted_input_features.npy',
 'block_0303_extracted_input_features.npy',
 'block_0304_extracted_input_features.npy',
 'block_0305_extracted_input_features.npy',
 'block_0306_extracted_input_features.npy']

In [8]:
# Okay, what next?

In [9]:
# Predict for test data? And also do a posthoc normalization step (considering the generic case to get the tassel densities per test image)

In [10]:
# get the image height and width
image_height = 768
image_width = 1024
print(image_height, image_width)

768 1024


In [11]:
# all_test_featurefile_names

In [12]:
# Do this for a single block of data?
loaded_test_featured = np.load(os.path.join(test_features_location, all_test_featurefile_names[0]))

In [13]:
loaded_test_featured.shape

(910, 13, 32)

In [14]:
pred_vals_test_im_0 = basemodel2_stage2.predict(loaded_test_featured)

29/29 [==============================] - 0s 3ms/step


2025-06-26 12:22:12.741617: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


In [15]:
pred_vals_test_im_0.shape

(910, 7)

In [16]:
# reshape the predicted value, get rid of the final dimension
pred_vals_test_im_0 = pred_vals_test_im_0.reshape(pred_vals_test_im_0.shape[0], pred_vals_test_im_0.shape[1])

In [17]:
pred_vals_test_im_0.shape

(910, 7)

In [18]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 30, kernel_size = 30):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [19]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'all_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_density']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_density']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_density']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_density']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'all_predicted_counts_basemodel2'
    # save this file
    final_df.to_csv(os.path.join(final_loc, csv_file_name.split(".")[0][-4:] + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

In [20]:
all_csv_files = os.listdir("all_true_counts")
all_csv_files.sort()

In [21]:
all_test_blocks = [file.split(".")[0].split("_")[1] for file in all_test_featurefile_names]
all_test_blocks.sort()

In [22]:
all_test_csv_files = [file for file in all_csv_files if file.split(".")[0].split("_")[-1] in all_test_blocks]
all_test_csv_files.sort()

In [23]:
all_test_csv_files

['true_counts_0103.csv',
 'true_counts_0104.csv',
 'true_counts_0105.csv',
 'true_counts_0106.csv',
 'true_counts_0201.csv',
 'true_counts_0202.csv',
 'true_counts_0205.csv',
 'true_counts_0206.csv',
 'true_counts_0302.csv',
 'true_counts_0303.csv',
 'true_counts_0304.csv',
 'true_counts_0305.csv',
 'true_counts_0306.csv']

In [24]:
preds_block_0103, metrics_0103, preds_df_0103 = get_final_forecasted_and_true_values(pred_vals_test_im_0, image_height, image_width, 30, 30, all_test_csv_files[0])

In [25]:
preds_block_0103

[44.504407778780056,
 52.307843008202795,
 56.476224494500016,
 35.84232101151778,
 35.801032233063836,
 32.46920074505831,
 20.524130851009318]

In [26]:
metrics_0103

[7.991442427798097,
 9.051245774896401,
 PearsonRResult(statistic=0.7719742900053086, pvalue=0.041994236442167175),
 -2.0593662935266677]

In [27]:
preds_df_0103

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.000661,44.504408
1,test_im_1,39.000001,52.307843
2,test_im_2,41.000000,56.476224
3,test_im_3,31.000000,35.842321
4,test_im_4,32.000000,35.801032
5,test_im_5,40.002086,32.469201
6,test_im_6,27.000176,20.524131


In [28]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_im_0, axis = 0)

array([44.50439 , 52.30787 , 56.476227, 35.84232 , 35.801037, 32.469196,
       20.524126], dtype=float32)

In [29]:
# note the values match

In [30]:
# Now do this for all the test blocks

Block 0104

In [31]:
all_test_featurefile_names[1]

'block_0104_extracted_input_features.npy'

In [32]:
# Do this for a single block of data?
loaded_test_features_0104 = np.load(os.path.join(test_features_location, all_test_featurefile_names[1]))

In [33]:
loaded_test_features_0104.shape

(910, 13, 32)

In [36]:
pred_vals_test_0104 = basemodel2_stage2.predict(loaded_test_features_0104)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0104 = pred_vals_test_0104.reshape(pred_vals_test_0104.shape[0], pred_vals_test_0104.shape[1])

29/29 [==============================] - 0s 3ms/step


In [37]:
pred_vals_test_0104.shape

(910, 7)

In [38]:
all_test_csv_files[1]

'true_counts_0104.csv'

In [39]:
preds_block_0104, metrics_0104, preds_df_0104 = get_final_forecasted_and_true_values(pred_vals_test_0104, image_height, image_width, 30, 30, all_test_csv_files[1])

In [40]:
preds_block_0104

[28.75642530140324,
 28.379009528382248,
 30.20407520849848,
 25.260056173036276,
 20.732626840934074,
 20.158657198554074,
 15.757032331352718]

In [41]:
metrics_0104

[12.275036371479064,
 14.209942081047679,
 PearsonRResult(statistic=-0.04714139403302965, pvalue=0.9200591797072206),
 -7.436404121841774]

In [42]:
preds_df_0104

,Test_image_name,True_density,Forecasted_value
0,test_im_0,33.000000,28.756425
1,test_im_1,30.000000,28.379010
2,test_im_2,39.000001,30.204075
3,test_im_3,40.000000,25.260056
4,test_im_4,40.998810,20.732627
5,test_im_5,42.169009,20.158657
6,test_im_6,30.005317,15.757032


In [43]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0104, axis = 0)

array([28.75641 , 28.379015, 30.204056, 25.260035, 20.732643, 20.158665,
       15.75702 ], dtype=float32)

Block 0105

In [44]:
all_test_featurefile_names[2]

'block_0105_extracted_input_features.npy'

In [45]:
# Do this for a single block of data?
loaded_test_features_0105 = np.load(os.path.join(test_features_location, all_test_featurefile_names[2]))

In [46]:
loaded_test_features_0105.shape

(910, 13, 32)

In [47]:
pred_vals_test_0105 = basemodel2_stage2.predict(loaded_test_features_0105)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0105 = pred_vals_test_0105.reshape(pred_vals_test_0105.shape[0], pred_vals_test_0105.shape[1])

29/29 [==============================] - 0s 2ms/step


In [48]:
pred_vals_test_0105.shape

(910, 7)

In [49]:
all_test_csv_files[2]

'true_counts_0105.csv'

In [50]:
preds_block_0105, metrics_0105, preds_df_0105 = get_final_forecasted_and_true_values(pred_vals_test_0105, image_height, image_width, 30, 30, all_test_csv_files[2])

In [51]:
preds_block_0105

[35.46057477629978,
 35.33654833419533,
 38.65208400272299,
 30.718621793815416,
 26.03057022946521,
 24.34921482902513,
 17.486303287778583]

In [52]:
metrics_0105

[10.995823873696123,
 11.980411953470531,
 PearsonRResult(statistic=0.8903624356332035, pvalue=0.007200380249647951),
 -0.5143013990187524]

In [53]:
preds_df_0105

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.000001,35.460575
1,test_im_1,46.002743,35.336548
2,test_im_2,58.000696,38.652084
3,test_im_3,41.000032,30.718622
4,test_im_4,41.001190,26.030570
5,test_im_5,36.000022,24.349215
6,test_im_6,23.000000,17.486303


In [54]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0105, axis = 0)

array([35.460556, 35.336533, 38.65206 , 30.718645, 26.030575, 24.349226,
       17.486307], dtype=float32)

Block 0106

In [55]:
all_test_featurefile_names[3]

'block_0106_extracted_input_features.npy'

In [56]:
# Do this for a single block of data?
loaded_test_features_0106 = np.load(os.path.join(test_features_location, all_test_featurefile_names[3]))

In [57]:
loaded_test_features_0106.shape

(910, 13, 32)

In [59]:
pred_vals_test_0106 = basemodel2_stage2.predict(loaded_test_features_0106)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0106 = pred_vals_test_0106.reshape(pred_vals_test_0106.shape[0], pred_vals_test_0106.shape[1])

29/29 [==============================] - 0s 2ms/step


In [60]:
pred_vals_test_0106.shape

(910, 7)

In [61]:
all_test_csv_files[3]

'true_counts_0106.csv'

In [62]:
preds_block_0106, metrics_0106, preds_df_0106 = get_final_forecasted_and_true_values(pred_vals_test_0106, image_height, image_width, 30, 30, all_test_csv_files[3])

In [63]:
preds_block_0106

[35.27444558717602,
 35.531645094580455,
 39.09495038844806,
 32.31839432041626,
 28.031517311190964,
 26.8248410577662,
 17.447966402748307]

In [64]:
metrics_0106

[11.06561161460642,
 13.17065326512865,
 PearsonRResult(statistic=0.12767319206031588, pvalue=0.7850174475125068),
 -13.553999047941339]

In [65]:
preds_df_0106

,Test_image_name,True_density,Forecasted_value
0,test_im_0,38.999667,35.274446
1,test_im_1,38.999989,35.531645
2,test_im_2,45.000000,39.094950
3,test_im_3,39.986887,32.318394
4,test_im_4,42.999997,28.031517
5,test_im_5,47.996502,26.824841
6,test_im_6,38.000000,17.447966


In [66]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0106, axis = 0)

array([35.27443 , 35.531628, 39.094925, 32.318413, 28.031525, 26.824844,
       17.447962], dtype=float32)

Block 0201

In [67]:
all_test_featurefile_names[4]

'block_0201_extracted_input_features.npy'

In [68]:
# Do this for a single block of data?
loaded_test_features_0201 = np.load(os.path.join(test_features_location, all_test_featurefile_names[4]))

In [69]:
loaded_test_features_0201.shape

(910, 13, 32)

In [70]:
pred_vals_test_0201 = basemodel2_stage2.predict(loaded_test_features_0201)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0201 = pred_vals_test_0201.reshape(pred_vals_test_0201.shape[0], pred_vals_test_0201.shape[1])

29/29 [==============================] - 0s 3ms/step


In [71]:
pred_vals_test_0201.shape

(910, 7)

In [72]:
all_test_csv_files[4]

'true_counts_0201.csv'

In [73]:
preds_block_0201, metrics_0201, preds_df_0201 = get_final_forecasted_and_true_values(pred_vals_test_0201, image_height, image_width, 30, 30, all_test_csv_files[4])

In [74]:
preds_block_0201

[39.50760083572868,
 44.476542161364506,
 49.59937886813712,
 35.67950109994795,
 34.24330569775651,
 32.08683736685772,
 18.723662893825974]

In [75]:
metrics_0201

[4.554603573692908,
 5.565723222645316,
 PearsonRResult(statistic=0.9397250082187577, pvalue=0.0016581627507745132),
 0.1424460692820373]

In [76]:
preds_df_0201

,Test_image_name,True_density,Forecasted_value
0,test_im_0,45.000217,39.507601
1,test_im_1,45.000040,44.476542
2,test_im_2,47.000001,49.599379
3,test_im_3,38.000000,35.679501
4,test_im_4,42.000041,34.243306
5,test_im_5,35.000000,32.086837
6,test_im_6,29.000000,18.723663


In [77]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0201, axis = 0)

array([39.50758 , 44.47653 , 49.599373, 35.6795  , 34.243305, 32.086838,
       18.723667], dtype=float32)

Block 0202

In [78]:
all_test_featurefile_names[5]

'block_0202_extracted_input_features.npy'

In [79]:
# Do this for a single block of data?
loaded_test_features_0202 = np.load(os.path.join(test_features_location, all_test_featurefile_names[5]))

In [80]:
loaded_test_features_0202.shape

(910, 13, 32)

In [81]:
pred_vals_test_0202 = basemodel2_stage2.predict(loaded_test_features_0202)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0202 = pred_vals_test_0202.reshape(pred_vals_test_0202.shape[0], pred_vals_test_0202.shape[1])

29/29 [==============================] - 0s 3ms/step


In [82]:
pred_vals_test_0202.shape

(910, 7)

In [83]:
all_test_csv_files[5]

'true_counts_0202.csv'

In [84]:
preds_block_0202, metrics_0202, preds_df_0202 = get_final_forecasted_and_true_values(pred_vals_test_0202, image_height, image_width, 30, 30, all_test_csv_files[5])

In [85]:
preds_block_0202

[32.88185615540826,
 35.855281127387016,
 39.98620313689689,
 27.266478413619723,
 24.51021060220502,
 18.276557268291924,
 17.07246325817445]

In [86]:
metrics_0202

[8.243454495814621,
 10.577662969350143,
 PearsonRResult(statistic=0.6800079900149156, pvalue=0.09278635654695408),
 -31.63353144756558]

In [87]:
preds_df_0202

,Test_image_name,True_density,Forecasted_value
0,test_im_0,17.999960,32.881856
1,test_im_1,21.000000,35.855281
2,test_im_2,23.000000,39.986203
3,test_im_3,20.999982,27.266478
4,test_im_4,21.000000,24.510211
5,test_im_5,18.000000,18.276557
6,test_im_6,18.000000,17.072463


In [88]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0202, axis = 0)

array([32.881855, 35.85528 , 39.986195, 27.266481, 24.510216, 18.27656 ,
       17.072468], dtype=float32)

Block 0205

In [89]:
all_test_featurefile_names[6]

'block_0205_extracted_input_features.npy'

In [90]:
# Do this for a single block of data?
loaded_test_features_0205 = np.load(os.path.join(test_features_location, all_test_featurefile_names[6]))

In [91]:
loaded_test_features_0205.shape

(910, 13, 32)

In [92]:
pred_vals_test_0205 = basemodel2_stage2.predict(loaded_test_features_0205)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0205 = pred_vals_test_0205.reshape(pred_vals_test_0205.shape[0], pred_vals_test_0205.shape[1])

29/29 [==============================] - 0s 3ms/step


In [93]:
pred_vals_test_0205.shape

(910, 7)

In [94]:
all_test_csv_files[6]

'true_counts_0205.csv'

In [95]:
preds_block_0205, metrics_0205, preds_df_0205 = get_final_forecasted_and_true_values(pred_vals_test_0205, image_height, image_width, 30, 30, all_test_csv_files[6])

In [96]:
preds_block_0205

[36.86941670576867,
 43.69979718258897,
 47.43919706634313,
 32.60287151916709,
 31.854072018265917,
 30.840884958845056,
 18.541735081424477]

In [97]:
metrics_0205

[7.48994050913875,
 8.478294105525855,
 PearsonRResult(statistic=0.8591609455515815, pvalue=0.0132337221104085),
 -3.3806041518478844]

In [98]:
preds_df_0205

,Test_image_name,True_density,Forecasted_value
0,test_im_0,44.000000,36.869417
1,test_im_1,42.000001,43.699797
2,test_im_2,45.000000,47.439197
3,test_im_3,43.000000,32.602872
4,test_im_4,39.000000,31.854072
5,test_im_5,40.999915,30.840885
6,test_im_6,31.999656,18.541735


In [99]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0205, axis = 0)

array([36.86942 , 43.69982 , 47.439217, 32.60286 , 31.854065, 30.840889,
       18.541739], dtype=float32)

Block 0206

In [100]:
all_test_featurefile_names[7]

'block_0206_extracted_input_features.npy'

In [101]:
# Do this for a single block of data?
loaded_test_features_0206 = np.load(os.path.join(test_features_location, all_test_featurefile_names[7]))

In [102]:
loaded_test_features_0206.shape

(910, 13, 32)

In [103]:
pred_vals_test_0206 = basemodel2_stage2.predict(loaded_test_features_0206)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0206 = pred_vals_test_0206.reshape(pred_vals_test_0206.shape[0], pred_vals_test_0206.shape[1])

29/29 [==============================] - 0s 3ms/step


In [104]:
pred_vals_test_0206.shape

(910, 7)

In [105]:
all_test_csv_files[7]

'true_counts_0206.csv'

In [106]:
preds_block_0206, metrics_0206, preds_df_0206 = get_final_forecasted_and_true_values(pred_vals_test_0206, image_height, image_width, 30, 30, all_test_csv_files[7])

In [107]:
preds_block_0206

[38.01684200557625,
 40.20831118214585,
 43.974054271572854,
 32.765780424422175,
 29.493099757129485,
 27.294202384988967,
 17.658416038623614]

In [108]:
metrics_0206

[2.8059648003987974,
 3.298753254992064,
 PearsonRResult(statistic=0.9409741401772141, pvalue=0.0015746597032096517),
 0.8624256723121777]

In [109]:
preds_df_0206

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.999838,38.016842
1,test_im_1,41.998810,40.208311
2,test_im_2,39.000067,43.974054
3,test_im_3,32.000003,32.765780
4,test_im_4,25.000352,29.493100
5,test_im_5,23.000040,27.294202
6,test_im_6,18.000000,17.658416


In [110]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0206, axis = 0)

array([38.01685 , 40.208313, 43.974083, 32.76579 , 29.493101, 27.294209,
       17.658422], dtype=float32)

Block 0302

In [111]:
all_test_featurefile_names[8]

'block_0302_extracted_input_features.npy'

In [112]:
# Do this for a single block of data?
loaded_test_features_0302 = np.load(os.path.join(test_features_location, all_test_featurefile_names[8]))

In [113]:
loaded_test_features_0302.shape

(910, 13, 32)

In [114]:
pred_vals_test_0302 = basemodel2_stage2.predict(loaded_test_features_0302)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0302 = pred_vals_test_0302.reshape(pred_vals_test_0302.shape[0], pred_vals_test_0302.shape[1])

29/29 [==============================] - 0s 3ms/step


In [115]:
pred_vals_test_0302.shape

(910, 7)

In [116]:
all_test_csv_files[8]

'true_counts_0302.csv'

In [117]:
preds_block_0302, metrics_0302, preds_df_0302 = get_final_forecasted_and_true_values(pred_vals_test_0302, image_height, image_width, 30, 30, all_test_csv_files[8])

In [118]:
preds_block_0302

[42.909962923578476,
 47.91137762769495,
 52.41201688743617,
 36.41419952678751,
 36.01125113475129,
 32.531410450342634,
 19.800681873287544]

In [119]:
metrics_0302

[8.864514867691259,
 10.760208939048074,
 PearsonRResult(statistic=0.8673803192232106, pvalue=0.011439311153699857),
 -3.4265080723344665]

In [120]:
preds_df_0302

,Test_image_name,True_density,Forecasted_value
0,test_im_0,49.000000,42.909963
1,test_im_1,49.000005,47.911378
2,test_im_2,53.999570,52.412017
3,test_im_3,50.042349,36.414200
4,test_im_4,43.000009,36.011251
5,test_im_5,48.000572,32.531410
6,test_im_6,36.999999,19.800682


In [121]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0302, axis = 0)

array([42.909977, 47.911392, 52.412006, 36.41418 , 36.011257, 32.531425,
       19.80067 ], dtype=float32)

Block 0303

In [122]:
all_test_featurefile_names[9]

'block_0303_extracted_input_features.npy'

In [123]:
# Do this for a single block of data?
loaded_test_features_0303 = np.load(os.path.join(test_features_location, all_test_featurefile_names[9]))

In [124]:
loaded_test_features_0303.shape

(910, 13, 32)

In [125]:
pred_vals_test_0303 = basemodel2_stage2.predict(loaded_test_features_0303)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0303 = pred_vals_test_0303.reshape(pred_vals_test_0303.shape[0], pred_vals_test_0303.shape[1])

29/29 [==============================] - 0s 3ms/step


In [126]:
pred_vals_test_0303.shape

(910, 7)

In [127]:
all_test_csv_files[9]

'true_counts_0303.csv'

In [128]:
preds_block_0303, metrics_0303, preds_df_0303 = get_final_forecasted_and_true_values(pred_vals_test_0303, image_height, image_width, 30, 30, all_test_csv_files[9])

In [129]:
preds_block_0303

[39.75003645861328,
 45.33833080406822,
 49.616328101697675,
 33.61829201049238,
 30.951984741302788,
 28.499566955336864,
 19.617083167550632]

In [130]:
metrics_0303

[5.40958672794076,
 6.002221333650227,
 PearsonRResult(statistic=0.8615689655610693, pvalue=0.012692469437433294),
 0.4251674853517652]

In [131]:
preds_df_0303

,Test_image_name,True_density,Forecasted_value
0,test_im_0,49.000171,39.750036
1,test_im_1,46.000007,45.338331
2,test_im_2,42.999982,49.616328
3,test_im_3,38.999993,33.618292
4,test_im_4,36.025884,30.951985
5,test_im_5,36.000000,28.499567
6,test_im_6,23.000000,19.617083


In [132]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0303, axis = 0)

array([39.750023, 45.338333, 49.616337, 33.618298, 30.951975, 28.499548,
       19.617102], dtype=float32)

Block 0304

In [133]:
all_test_featurefile_names[10]

'block_0304_extracted_input_features.npy'

In [134]:
# Do this for a single block of data?
loaded_test_features_0304 = np.load(os.path.join(test_features_location, all_test_featurefile_names[10]))

In [135]:
loaded_test_features_0304.shape

(910, 13, 32)

In [136]:
pred_vals_test_0304 = basemodel2_stage2.predict(loaded_test_features_0304)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0304 = pred_vals_test_0304.reshape(pred_vals_test_0304.shape[0], pred_vals_test_0304.shape[1])

29/29 [==============================] - 0s 3ms/step


In [137]:
pred_vals_test_0304.shape

(910, 7)

In [138]:
all_test_csv_files[10]

'true_counts_0304.csv'

In [139]:
preds_block_0304, metrics_0304, preds_df_0304 = get_final_forecasted_and_true_values(pred_vals_test_0304, image_height, image_width, 30, 30, all_test_csv_files[10])

In [140]:
preds_block_0304

[34.69418797181418,
 36.66457626770494,
 40.45668163904154,
 28.092437238574874,
 27.16496059058042,
 23.555624039034452,
 16.785480343627377]

In [141]:
metrics_0304

[7.369723112427847,
 8.47306966976148,
 PearsonRResult(statistic=0.8351985959503752, pvalue=0.019337288497273727),
 -0.9629647829115502]

In [142]:
preds_df_0304

,Test_image_name,True_density,Forecasted_value
0,test_im_0,37.000000,34.694188
1,test_im_1,41.002057,36.664576
2,test_im_2,42.999998,40.456682
3,test_im_3,41.999955,28.092437
4,test_im_4,38.000000,27.164961
5,test_im_5,34.000000,23.555624
6,test_im_6,24.000000,16.785480


In [143]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0304, axis = 0)

array([34.694218, 36.664562, 40.45666 , 28.092442, 27.16497 , 23.555607,
       16.785479], dtype=float32)

Block 0305

In [144]:
all_test_featurefile_names[11]

'block_0305_extracted_input_features.npy'

In [145]:
# Do this for a single block of data?
loaded_test_features_0305 = np.load(os.path.join(test_features_location, all_test_featurefile_names[11]))

In [146]:
loaded_test_features_0305.shape

(910, 13, 32)

In [148]:
pred_vals_test_0305 = basemodel2_stage2.predict(loaded_test_features_0305)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0305 = pred_vals_test_0305.reshape(pred_vals_test_0305.shape[0], pred_vals_test_0305.shape[1])

29/29 [==============================] - 0s 2ms/step


In [149]:
pred_vals_test_0305.shape

(910, 7)

In [150]:
all_test_csv_files[11]

'true_counts_0305.csv'

In [151]:
preds_block_0305, metrics_0305, preds_df_0305 = get_final_forecasted_and_true_values(pred_vals_test_0305, image_height, image_width, 30, 30, all_test_csv_files[11])

In [152]:
preds_block_0305

[39.72654619858501,
 42.765905686004366,
 47.2668282112952,
 33.13719050166213,
 30.690789221491755,
 27.507554640664566,
 17.70967360202968]

In [153]:
metrics_0305

[5.07655859073206,
 5.86714524726492,
 PearsonRResult(statistic=0.7828044885064415, pvalue=0.03741823211248594),
 0.3771245667491081]

In [154]:
preds_df_0305

,Test_image_name,True_density,Forecasted_value
0,test_im_0,46.000000,39.726546
1,test_im_1,35.000000,42.765906
2,test_im_2,37.000000,47.266828
3,test_im_3,30.000652,33.137191
4,test_im_4,35.001190,30.690789
5,test_im_5,28.999994,27.507555
6,test_im_6,20.000018,17.709674


In [155]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0305, axis = 0)

array([39.726574, 42.76587 , 47.26688 , 33.137215, 30.690796, 27.507534,
       17.709671], dtype=float32)

Block 0306

In [156]:
all_test_featurefile_names[12]

'block_0306_extracted_input_features.npy'

In [157]:
# Do this for a single block of data?
loaded_test_features_0306 = np.load(os.path.join(test_features_location, all_test_featurefile_names[12]))

In [158]:
loaded_test_features_0306.shape

(910, 13, 32)

In [ ]:
pred_vals_test_0306 = basemodel2_stage2.predict(loaded_test_features_0306)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0306 = pred_vals_test_0306.reshape(pred_vals_test_0306.shape[0], pred_vals_test_0306.shape[1])

In [ ]:
pred_vals_test_0306.shape

In [ ]:
all_test_csv_files[12]

In [ ]:
preds_block_0306, metrics_0306, preds_df_0306 = get_final_forecasted_and_true_values(pred_vals_test_0306, image_height, image_width, 30, 30, all_test_csv_files[12])

In [ ]:
preds_block_0306

In [ ]:
metrics_0306

In [ ]:
preds_df_0306

In [ ]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0306, axis = 0)

In [ ]:
# Get the average metrics also on the entire test data? For future use maybe

In [ ]:
# Since we have the preds and true values all stored here  - "all_predicted_counts", let's use that to get the final metrics on the entire test space.

In [ ]:
test_true_and_preds_path = "all_predicted_counts"

In [ ]:
all_contents_here = os.listdir(test_true_and_preds_path)
all_contents_here.sort()

In [ ]:
all_contents_here

In [ ]:
all_csv_files = [file for file in all_contents_here if file.split(".")[-1] == "csv"]

In [ ]:
# all_csv_files

In [ ]:
# load all csv files?
csv_files_all = []
for file in all_csv_files:
    loaded_file = pd.read_csv(os.path.join(test_true_and_preds_path, file))
    csv_files_all.append(loaded_file)

In [ ]:
all_test_preds_and_true_densities = pd.concat(csv_files_all)

In [ ]:
all_test_preds_and_true_densities.head()

In [ ]:
all_test_preds_and_true_densities.shape

In [ ]:
# mae
all_test_mae = mean_absolute_error(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
all_test_mae

In [ ]:
# mse
all_test_mse = mean_squared_error(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
np.sqrt(all_test_mse)

In [ ]:
# r2
all_test_r2_score = r2_score(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
all_test_r2_score

In [ ]:
# pearsonr
all_test_pearson = pearsonr(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
all_test_pearson[0]